# Chapter 8, Part 1: RAG-Based Recommender

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kimfalk/modern-recommender-systems/blob/main/notebooks/chapter-08/01_rag_recommender.ipynb)

This notebook builds a Retrieval-Augmented Generation (RAG) recommender system.
We reuse the BERT embeddings and FAISS index from Chapters 5–6 and add an LLM
reasoning layer on top.

**What you'll build:**
- Vector store setup with movie embeddings
- Hybrid retrieval (content + collaborative, merged via reciprocal rank fusion)
- RAG recommendation chain
- Query rewriting and HyDE
- Cold-start item addition

**Prerequisites:** Run the Chapter 5 (ANN search) and Chapter 6 (semantic IDs) notebooks first,
or use the pre-computed embeddings loaded below.

In [ ]:
# Cell 1: Environment Setup
from recsys.utils.colab import setup_colab_environment, get_data_path, check_gpu

setup_colab_environment()
check_gpu()

In [ ]:
# Cell 2: Imports
import os
os.environ["OMP_NUM_THREADS"] = "1"

import numpy as np
import pandas as pd
import faiss
import json
from pathlib import Path
from sentence_transformers import SentenceTransformer

from recsys.data.loaders import (
    load_movielens,
    load_movielens_links,
    load_tmdb_movie_descriptions,
)
from recsys.fourstage_recsys.retrieval.itemknn_retrieval import ItemKNNRetrieval
from recsys.agentic.llm_client import LLMClient
from recsys.agentic.hybrid_retriever import HybridRetriever

DATA_PATH = get_data_path()

## LLM Backend Setup

The RAG chain needs an LLM for reasoning. Choose one backend:

- **API** (recommended): Uses OpenAI-compatible API. Set your API key below.
- **Local**: Uses a HuggingFace model. No API key needed but slower and lower quality for agent reasoning.

Both produce the same interface — the rest of the notebook works identically.

In [ ]:
%pip install accelerate

In [ ]:
# Cell 3: LLM Setup — pick ONE of these

# Option A: API backend (recommended)
# import os
# llm = LLMClient(
#   backend="api",
#   api_key=os.environ.get("OPENAI_API_KEY"),
#   model="gpt-4o-mini"
# )

#Option B: Local backend (no API key needed)

llm = LLMClient(
  backend="local",
  model="microsoft/Phi-3-mini-4k-instruct"
)

## 1. Load Data and Embeddings

We load the MovieLens dataset and compute embeddings using the same
`all-MiniLM-L6-v2` encoder from Chapter 6. If you have pre-computed
embeddings saved from Chapter 6, load those instead.

In [ ]:
# Cell 4: Load movie data
ratings, movies = load_movielens(
  dataset='ml-25m', data_dir=DATA_PATH
)

print(f"Ratings: {len(ratings):,}")
print(f"Movies: {len(movies):,}")

In [ ]:

import os
from dotenv import load_dotenv

load_dotenv()
tmdb_api_key = os.getenv("TMDB_API_KEY")

DATA_PATH = get_data_path()
ratings, movies = load_movielens("ml-25m", data_dir=DATA_PATH)
links = load_movielens_links("ml-25m", data_dir=DATA_PATH)

descriptions = load_tmdb_movie_descriptions(
    links=links,
    api_key=tmdb_api_key,
    data_dir=DATA_PATH,
    cache_filename="movielens_descriptions.csv",
    force_refresh=False,
)

desc_df = pd.DataFrame.from_dict(descriptions, orient='index')
desc_df.reset_index(names='movieId', inplace=True)
desc_df['movieId'] = desc_df['movieId'].astype(str)

movies = movies.merge(desc_df[['movieId', 'overview']], on='movieId', how='left')
movies['overview'] = movies['overview'].fillna('')
movies['content'] = movies['title'] + ' ' + movies['genres'] + ' ' + movies['overview']

In [ ]:
# Cell 6: Compute or load embeddings
EMBEDDINGS_PATH = Path(DATA_PATH) / "movie_embeddings.npy"

if EMBEDDINGS_PATH.exists():
  print("Loading pre-computed embeddings...")
  embeddings = np.load(EMBEDDINGS_PATH)
else:
  print("Computing embeddings (this takes a few minutes)...")
  encoder = SentenceTransformer('all-MiniLM-L6-v2')  #A
  
  # Combine title + genres + overview for richer embeddings
  texts = [
    f"{row['title']} {row.get('genres', '')} {row.get('overview', '')}"
    for _, row in movies.iterrows()
  ]
  embeddings = encoder.encode(
    texts, show_progress_bar=True, batch_size=64
  )
  np.save(EMBEDDINGS_PATH, embeddings)
  print(f"Saved embeddings to {EMBEDDINGS_PATH}")

print(f"Embeddings shape: {embeddings.shape}")  #B
#A Same encoder as Chapter 6
#B Should be (num_movies, 384)

## 2. Build the FAISS Index

We build a FAISS index from the embeddings. This is the same pattern
as Chapter 5's ANN search — the only difference is that we'll query it
with natural-language embeddings instead of item embeddings.

In [ ]:
# Cell 7: Build FAISS index
embeddings = np.ascontiguousarray(embeddings.astype(np.float32)) #A
dim = embeddings.shape[1]  # 384
index = faiss.IndexFlatIP(dim)  #B

print(f"Adding items to FAISS index")
# Normalize for cosine similarity via inner product
faiss.normalize_L2(embeddings)
index.add(embeddings.astype(np.float32))

print(f"FAISS index built: {index.ntotal} vectors, dim={dim}")
#A FAISS requires contiguous float32 arrays
#B IndexFlatIP with L2-normalized vectors = cosine similarity

## 3. Hybrid Retriever

The `HybridRetriever` composes content-based retrieval (FAISS) with
collaborative filtering (ItemKNN from Chapter 3). It merges results
using reciprocal rank fusion.

In [ ]:
# Cell 8: Build ItemKNN for collaborative signal
# Use a sample of ratings for speed
ratings_sample = ratings.sample(
  n=min(500_000, len(ratings)), random_state=42
)
item_knn = ItemKNNRetrieval(ratings_sample)
print(f"ItemKNN built from {len(ratings_sample):,} ratings")

In [ ]:
# Cell 9: Build hybrid retriever
retriever = HybridRetriever(
  faiss_index=index,
  embeddings=embeddings,
  movies_df=movies.reset_index(drop=True),
  item_knn=item_knn
)

# Quick test: search by text
results = retriever.search("dark sci-fi thriller", k=5)
for r in results:
  print(f"  {r['title']} [{r['genres']}] score={r['score']:.3f}")

## 4. RAG Recommendation Chain

This is the core pattern: retrieve candidates, assemble them into a prompt
with user context, and let the LLM select and explain. The retriever handles
catalog grounding; the LLM handles reasoning.

In [ ]:
# Cell 10: RAG recommendation function
def recommend_with_rag(query, retriever, llm,
                       user_id=None, k=15):
  """End-to-end RAG recommendation."""
  # Step 1: Retrieve candidates
  candidates = retriever.search(
    query, user_id=user_id, k=k
  )
  
  # Step 2: Build prompt
  candidates_text = "\n".join(
    f"- {c['title']} ({c.get('year', 'N/A')}) "
    f"[{c.get('genres', '')}]: "
    f"{c.get('overview', '')[:150]}"
    for c in candidates
  )
  
  system_prompt = """You are a movie recommendation assistant.
Given a user's request and candidate movies, select the 
best 3-5 matches and explain why each fits.

Rules:
- Only recommend movies from the candidates list
- Explain your reasoning for each pick
- If none fit well, say so honestly
- Keep explanations concise but specific"""
  
  user_message = (
    f"Request: {query}\n\n"
    f"Candidates:\n{candidates_text}"
  )
  
  # Step 3: Generate
  response = llm.generate(
    system_prompt=system_prompt,
    user_message=user_message
  )
  return response, candidates

In [ ]:
# Cell 11: Test the RAG chain
response, candidates = recommend_with_rag(
  "A sci-fi movie from the 1990s similar to Dune",
  retriever, llm
)
print(response)

In [ ]:
# Cell 12: Try an abstract query
response, _ = recommend_with_rag(
  "Something to watch when I'm feeling nostalgic on a rainy Sunday",
  retriever, llm
)
print(response)

## 5. Query Rewriting and HyDE

Abstract queries like "something nostalgic" don't embed well because
no movie description says "I am nostalgic." HyDE (Hypothetical Document
Embeddings) fixes this by generating a hypothetical movie description
that *would* match, then using that description as the search query.

In [ ]:
# Cell 13: HyDE implementation
def hyde_retrieval(llm, retriever, query, k=10):
  """Retrieve using a hypothetical document."""
  # Step 1: Generate hypothetical movie description
  hypothetical = llm.generate(
    system_prompt=(
      "You are a movie database. Given a user's request, "
      "write a detailed description of a movie that would "
      "perfectly match. Include genre, tone, themes, and "
      "plot elements. The movie does not need to be real."
    ),
    user_message=query
  )
  print(f"HyDE generated description:\n{hypothetical}\n")
  
  # Step 2: Use the description as search query
  results = retriever.search(hypothetical, k=k)
  return results, hypothetical

In [ ]:
# Cell 14: Compare direct retrieval vs HyDE
query = "something nostalgic for a rainy Sunday"

print("=== Direct retrieval ===")
direct = retriever.search(query, k=5)
for r in direct:
  print(f"  {r['title']} (score={r['score']:.3f})")

print("\n=== HyDE retrieval ===")
hyde_results, hyp = hyde_retrieval(llm, retriever, query, k=5)
for r in hyde_results:
  print(f"  {r['title']} (score={r['score']:.3f})")

## 6. Cold Start: Adding New Items

With RAG, a new item just needs to be embedded and added to the index.
No retraining, no vocabulary extension, no new semantic IDs. Compare this
to Chapter 7, where a new item required running the full Chapter 6 pipeline.

In [ ]:
# Cell 15: Add a new movie and search for it immediately
items_before = retriever.index.ntotal

retriever.add_item(
  title="Galactic Gardeners (2025)",
  genres="Sci-Fi|Comedy|Animation",
  overview=(
    "In a future where Earth's plants have gone extinct, "
    "a ragtag crew of space botanists must travel to "
    "distant planets to collect seeds and rebuild Earth's "
    "ecosystems. A heartwarming adventure about hope, "
    "teamwork, and really stubborn cacti."
  ),
  movie_id=999999
)

print(f"Index grew: {items_before} → {retriever.index.ntotal}")

# Search for it
results = retriever.search("animated space adventure about plants", k=5)
for r in results:
  print(f"  {r['title']} (score={r['score']:.3f})")

## Summary

This notebook built a RAG-based recommender:

1. **FAISS index** from the same all-MiniLM-L6-v2 embeddings used in Chapter 6
2. **HybridRetriever** combining content (FAISS) and collaborative (ItemKNN) retrieval with RRF
3. **RAG chain** where the retriever grounds recommendations and the LLM reasons and explains
4. **HyDE** for handling abstract queries that don't embed well directly
5. **Cold start** solved by embedding and adding — no retraining needed

The next notebook (Part 2) adds the agent loop, memory, and conversational capabilities.